# End-to-end simulation of observation and pipeline reduction of HR 8799 d

<span style="color:red">The HISPEC DRP is in active development. Please submit issues and suggestions on the [github](https://github.com/oirlab/HISPEC_DRP).</span>


This notebook shows an example of simulating a single exposure of an off guiding axis observation of HR 8799 d. 

The notebook simulates a BSPEC rate map using a combination of a Phoenix template (approximately similar to HR 8799 A) and Sonora bobcat template (approximately similar to HR 8799 d). The rate map is used to build a `RampModel`, then run `Stage0Pipeline` → `Stage1Pipeline` → `Stage2Pipeline`.

---

## Setup

**Environment variables** (set these in your shell / notebook kernel before running):

| Variable | Required when | Purpose |
|---|---|---|
| `KOA_CALIBRATION_CACHE` | Always | Local KOA calibration cache directory (and SQLite DB) |
| `KECK_OBSERVER_EMAIL` | Syncing / downloading from KOA | Keck Observer account email |
| `KECK_OBSERVER_PASSWORD` | Syncing / downloading from KOA | Keck Observer account password |

If `KOA_CALIBRATION_CACHE` is unset, the setup cell sets it under `OUT`.  
The KOA cell below syncs metadata from remote KOA and downloads any **BSPEC** calibration files that are not already in the cache.


In [ ]:
import os
import hispecdrp

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

# OUTPUT DIRECTORY — update this to your desired output directory
OUT = "/paht/to/hr8799d_example"
os.makedirs(OUT, exist_ok=True)

os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    os.path.join(OUT, "cals", "KOA_CALIBRATION_CACHE"),
)
print("output:", OUT)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])

## 1. Sync / verify KOA caldb (BSPEC)

In [ ]:
from hispecdrp.calibrations import verify_koa_calibrations

# Requires env vars: KOA_CALIBRATION_CACHE, KECK_OBSERVER_EMAIL, KECK_OBSERVER_PASSWORD
SPECTROGRAPH = "BSPEC"
summary = verify_koa_calibrations(SPECTROGRAPH, ignore_missing_cal_types=["lfc_wavesol"])

print(
    f"{summary['spectrograph']} caldb: {summary['n_records']} record(s); "
    f"downloaded {summary['n_downloaded']}/{summary['n_missing']} missing file(s)"
)
print("types:", summary["cal_types"])
if summary["downloaded"]:
    print("downloaded:", summary["downloaded"])
if summary["failed"]:
    print("failed:", summary["failed"])

## 3. Creating input spectra from templates

In the next cells, we will normalize the planet spectra to the correct magnitude, and add the stellar spectrum as a speckle at the expected contrast for the Keck PSF. We will also create the spectrum for a non-science speckle fiber which only contains a reduced portion of the stellar flux.

First, load the template spectra and normalize to approximate J band magnitudes.

In [ ]:
hr8799A_magnitude_J_vega = 5.38
hr8799A_magnitude_J_AB = hr8799A_magnitude_J_vega + 0.91

hr8799d_magnitude_J_vega = 15.61 # Zurlo et al. 2016
hr8799d_magnitude_J_AB = hr8799d_magnitude_J_vega + 0.91

from hispecdrp.primitives.utils.spectral_utils import load_btsettl_spectrum
from hispecdrp.simulations.e2e_support import templates_data_dir

template_dir = templates_data_dir()+'/'
hr8799A_template_filename = 'lte07400-4.00-0.5.PHOENIX-ACES-AGSS-COND-2011-HiRe.h5'
hr8799d_template_filename = 'sp_t1100g178nc_m+0.5.h5'

hr8799A_wvs, hr8799A_flam = load_btsettl_spectrum(
    filepath=template_dir+hr8799A_template_filename,
    mag_ab=hr8799A_magnitude_J_AB,
    filter_band='J'
)

hr8799d_wvs, hr8799d_flam = load_btsettl_spectrum(
    filepath=template_dir+hr8799d_template_filename,
    mag_ab=hr8799d_magnitude_J_AB,
    filter_band='J'
)

Now we will scale the stellar spectrum based on its separation from the star in the science and speckle fibers. A helper function is provided to estimate the speckle contrast as a function of separation, but it is only a rough estimate. We will doppler shift the planetary and stellar spectrum separately here before adding them together.

In [ ]:
import numpy as np

def estimated_speckle_contrast_at_separation(separation):
    # derived from linear fit to Ruffio et al. 2026 figure 17, Keck/OSIRIS in K band
    # separation must be in arcseconds

    m = -2.480700421412601
    b = -9.47882696118136
    log_sep = np.log(separation)
    log_contrast = m * log_sep + b

    return np.exp(log_contrast)

In [ ]:
separation_hr8799d = 0.7 # arcseconds
separation_speckle_fiber = 0.7 # arcseconds

speckle_contrast_hr8799d = estimated_speckle_contrast_at_separation(separation_hr8799d)
speckle_contrast_speckle_fiber = estimated_speckle_contrast_at_separation(separation_speckle_fiber)

stellar_speckle_planet_flam = hr8799A_flam * speckle_contrast_hr8799d
stellar_speckle_fiber_flam =  hr8799A_flam * speckle_contrast_speckle_fiber

# fiber 1 spectrum is planet + stellar speckle, we will interpolate the planet spectrum to the stellar wavelength grid
fiber1_flam = np.interp(hr8799A_wvs, hr8799d_wvs, hr8799d_flam) + stellar_speckle_planet_flam
# fiber 2 spectrum is just stellar speckles
fiber2_flam = stellar_speckle_fiber_flam
# no fiber 3 spectrum needs to be constructed since it will just be sky


In [ ]:
fiber1_flam.shape, hr8799A_wvs.shape

### 4 Simulated rate map from 1D spectra → `RampModel`

We will construct a dictionary of the spectra, RVs, and other inputs for each fiber, create an idealized rate map, which will then be converted to an initial ramp cube with artificial detector and poisson noise. Because HR 8779 d is near edge on, we will make the simplifying assumption that is has an identical radial velocity to the star. If not using this assumption, we would have to apply RVs separately to the template spectra before combining, and not use the included RV functionality below. 

1. Fiber 1 is the on the planet, which includes stellar speckles, and sky. 
2. Fiber 2 is a sample of the stellar speckles. Here we assume the speckles are the same as on fiber 1, but of course this will generally be the case. 
3. Fiber 3 is background sky flux.

What this simulation does **not** include:
- telluric injection/correction
- wavelength solution determination (here the wave solution is derived from a ZEMAX instrument model)
- atmospheric variability of any kind

In [ ]:
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum

FIBERS = [1,2,3]
READ_TIME = 30 # seconds

# Injected stellar RV (km/s). Default path densifies the 1D template, Doppler-shifts
# it, paints with ZEMAX, and attaches the fixed packaged line-ZP WAVE (no WAVE×(1+z)).
STELLAR_RV = -10.41

exposure_time = 900 # seconds
n_reads = int(exposure_time/READ_TIME)

# Create a UTR map from a controlled seed and template spectrum
# The user can normalize to a specific magnitude (AB mag) and 2MASS filter
ramp, cals, rate_map = create_ramp_from_spectrum(
    OUT,
    spectrograph=SPECTROGRAPH,
    fibers=FIBERS,
    fiber_spectra={
        1:(hr8799A_wvs, fiber1_flam),
        2:(hr8799A_wvs, fiber2_flam),
    },
    filter_band="J",  # BSPEC (yJ)
    n_reads=n_reads,
    seed=0,
    radial_velocity_km_s=STELLAR_RV, # no dictionary needed since it is the same for both fibers
    barycentric_velocity_km_s=0.0,
    inject_rv_via="spectrum",
    sky_filename='mk_skybg_zm_1.0_1.0_ph.dat',
    apply_sky=True,
    apply_e2e_throughput=True,
    save_ramp=True,
    overwrite_ramp=True,
    readtime=READ_TIME,
)

print(f"fibers: {FIBERS}")
print(f"ramp type: {ramp.meta.model_type}")
print(f"ramp filename: {ramp.meta.filename}")
print(f"ramp data shape: {ramp.data.shape}")
print(f"Stage-0 cals: {sorted(cals)}")
print(f"rate_map shape: {rate_map.shape}")
print(f"wavesol note: {getattr(getattr(cals.get('wavesol'), 'meta', None), 'note', None)}")

### 5 Stage 0 → 1 → 2 via pipeline classes

In [ ]:
from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline

l0 = Stage0Pipeline(output_dir=OUT).apply(ramp)

do_fibers = [str(f) for f in FIBERS]
# Fixed packaged line-ZP WAVE from create_ramp_from_spectrum (unscaled for spectrum paint)
stage1_args = {
    "spec_extract": {"do_fibers": do_fibers},
    "set_wavesol": {"wavesol": cals["wavesol"]},
}
l1 = Stage1Pipeline(output_dir=OUT, prim_args=stage1_args).apply(l0)

injected_m_s = float(STELLAR_RV) * 1000.0
l2 = Stage2Pipeline(
    output_dir=OUT,
    prim_args={
        "blaze_corr": {"do_fibers": do_fibers},
        "stellar_rv": {
            "ccf_input_velocity": injected_m_s,
            "do_fibers": [1],
            "save_preview": True,
            "preview_write_png": False,
        },
        "sky_sub": {"run": True},
    },
).apply(l1)

print(f"L0 type: {l0.meta.model_type}")
print(f"L0 filename: {l0.meta.filename}")
print(f"L1 type: {l1.meta.model_type}")
print(f"L1 filename: {l1.meta.filename}")
print(f"L2 type: {l2.meta.model_type}")
print(f"L2 filename: {l2.meta.filename}")

In [ ]:
dir(l2)

In [ ]:
l2.traces

In [ ]:
interactive_spectrum_plot(l2,keys=['spec'])

In [ ]:
"""
Interactive spectrum viewer -- pure Plotly, no ipywidgets/anywidget required.

Works with an object that has a dict attribute keyed by fiber number
(e.g. 1, 2, 3), where each value is itself a dict of 2D arrays shaped
(n_orders, n_pixels) -- including a 'wave' array giving wavelength per
pixel per order, e.g.:

    obj.data = {
        1: {'spec': ..., 'spec_err': ..., 'wave': ..., 'spec_dq': ..., ...},
        2: {...},
        3: {...},
    }

Everything is done with plotly's native `updatemenus` (dropdown, picks
fiber+key) and `sliders` (picks order) -- both are client-side JS, so this
works in a plain Jupyter notebook, JupyterLab, or as a standalone HTML file
(fig.write_html(...)) with no extra runtime/kernel comm required.

Requires: plotly only.
"""

import plotly.graph_objects as go


def interactive_spectrum_plot(obj, data_attr='traces', wave_key='wave',
                               keys=None, use_webgl=False):
    """
    Build a pure-plotly interactive figure to browse a spectral data cube.

    Parameters
    ----------
    obj : object
        Object holding the dict attribute, e.g. obj.data[fiber][key].
    data_attr : str
        Name of the dict attribute on obj (default 'data').
    wave_key : str
        Key that holds the wavelength array (default 'wave').
    keys : list[str] or None
        Which keys to include in the dropdown. If None, auto-detects all
        keys whose array shape matches `wave`. Restricting this list is
        recommended -- see the performance note below.
    use_webgl : bool
        Use Scattergl instead of Scatter (faster for large arrays, but
        heavier to instantiate many traces of). Default False.

    Returns
    -------
    plotly.graph_objects.Figure

    Note on performance
    --------------------
    Because the order slider is implemented as a client-side `restyle`
    step, ALL traces' data must be embedded in the HTML for every slider
    step (there's no live Python backend to fetch data on demand). Total
    embedded size is roughly:

        n_fibers * n_keys * n_orders * n_pixels * 2 (x and y)

    For 3 fibers x 5 keys x 39 orders x 4096 pixels this is a few tens of
    MB of JSON -- fine, but slow to build/render if you include many keys.
    Pass an explicit, short `keys` list (e.g. ['spec', 'spec_err']) to keep
    this fast.
    """
    data_dict = getattr(obj, data_attr)
    fibers = sorted(data_dict.keys())
    sample = data_dict[fibers[0]]
    wave_shape = sample[wave_key].shape
    n_orders, n_pixels = wave_shape

    if keys is None:
        keys = sorted(
            k for k, v in sample.items()
            if k != wave_key and hasattr(v, 'shape') and v.shape == wave_shape
        )

    Scatter = go.Scattergl if use_webgl else go.Scatter

    # combo list defines trace order: (fiber, key) pairs
    combos = [(f, k) for f in fibers for k in keys]
    n_traces = len(combos)

    # --- build initial traces at order=0 ---
    fig = go.Figure()
    for i, (f, k) in enumerate(combos):
        fig.add_trace(Scatter(
            x=data_dict[f][wave_key][0],
            y=data_dict[f][k][0],
            mode='lines',
            name=f'Fiber {f} - {k}',
            visible=(i == 0),
        ))

    fig.update_layout(
        template='plotly_white',
        height=550,
        xaxis_title='Wavelength',
        yaxis_title=keys[0],
        title=f'Fiber {combos[0][0]} - {combos[0][1]} - Order 0',
        margin=dict(t=90),
    )

    # --- dropdown: pick (fiber, key) combo -> toggles trace visibility ---
    dropdown_buttons = []
    for i, (f, k) in enumerate(combos):
        visible = [j == i for j in range(n_traces)]
        dropdown_buttons.append(dict(
            method='update',
            label=f'Fiber {f} - {k}',
            args=[
                {'visible': visible},
                {'yaxis.title.text': k,
                 'title.text': f'Fiber {f} - {k}'},
            ],
        ))

    fig.update_layout(updatemenus=[dict(
        type='dropdown',
        buttons=dropdown_buttons,
        direction='down',
        showactive=True,
        x=0.0, xanchor='left',
        y=1.15, yanchor='top',
    )])

    # --- slider: pick order -> restyles x/y of ALL traces at once ---
    slider_steps = []
    for o in range(n_orders):
        x_all = [data_dict[f][wave_key][o] for (f, k) in combos]
        y_all = [data_dict[f][k][o] for (f, k) in combos]
        slider_steps.append(dict(
            method='restyle',
            label=str(o),
            args=[{'x': x_all, 'y': y_all}],
        ))

    fig.update_layout(sliders=[dict(
        active=0,
        currentvalue={'prefix': 'Order: '},
        pad={'t': 60},
        steps=slider_steps,
    )])

    return fig


# ---------------------------------------------------------------------
# Usage:
#
#   from interactive_spectrum_plot import interactive_spectrum_plot
#
#   fig = interactive_spectrum_plot(my_obj, keys=['spec', 'spec_err'])
#   fig.show()
#
#   # or save as a standalone HTML file (shareable, no Python needed to view):
#   fig.write_html('spectrum_viewer.html')
# ---------------------------------------------------------------------